# Persian Retrieval Benchmark

Evaluate a Sentence Transformers model on selected Persian MTEB retrieval tasks.
This notebook preserves the original test-split evaluation, FP16 model loading,
512-token maximum length, and encoding batch size of 64.

Run on a GPU runtime with internet access. Edit the model and task selection below.
The default model is Yalda Embedding; the default selection is the original single
HotpotQA task. Use `TASK_NAMES = ALL_TASK_NAMES.copy()` for the paper's eight-task suite.


## 1. Install and import dependencies


In [ ]:
!pip install mteb -q


In [ ]:
import os
from importlib.metadata import version

import mteb
import torch
from sentence_transformers import SentenceTransformer

for package in ("mteb", "sentence-transformers", "transformers", "torch"):
    print(f"{package}: {version(package)}")


## 2. Configure the model and tasks

Public models and datasets usually work without a token. When needed, provide
`HF_TOKEN` through the runtime's environment/secrets. Change `MODEL_ID` to evaluate
another baseline and `TASK_NAMES` to evaluate one or several tasks.


In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", "")
MODEL_ID = "Yaldat/Yalda-Embedding"
MAX_SEQUENCE_LENGTH = 512
ENCODE_BATCH_SIZE = 64
TRUST_REMOTE_CODE = True

ALL_TASK_NAMES = [
    "FiQA2018-Fa",
    "NFCorpus-Fa",
    "SynPerQARetrieval",
    "PersianWebDocumentRetrieval",
    "HotpotQA-FaHardNegatives",
    "MSMARCO-FaHardNegatives",
    "NQ-FaHardNegatives",
    "SciFact-Fa",
]
TASK_NAMES = ["HotpotQA-FaHardNegatives"]
# To evaluate the entire suite instead:
# TASK_NAMES = ALL_TASK_NAMES.copy()


## 3. Load the model and test tasks

Pooling and prompt behavior come from the loaded Sentence Transformers model and
MTEB's integration. This notebook does not override them with a shared query prompt.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for this FP16 benchmark.")

model = SentenceTransformer(
    MODEL_ID,
    model_kwargs={"torch_dtype": torch.float16},
    trust_remote_code=TRUST_REMOTE_CODE,
    token=HF_TOKEN or None,
)
model.max_seq_length = MAX_SEQUENCE_LENGTH
tasks = mteb.get_tasks(tasks=TASK_NAMES, eval_splits=["test"])

print(f"Model: {MODEL_ID}")
print(f"Tasks: {', '.join(TASK_NAMES)}")
print(f"Maximum length: {MAX_SEQUENCE_LENGTH} | Encoding batch size: {ENCODE_BATCH_SIZE}")


## 4. Evaluate


In [ ]:
results = mteb.evaluate(
    model,
    tasks=tasks,
    encode_kwargs={
        "batch_size": ENCODE_BATCH_SIZE,
        "show_progress_bar": True,
    },
)


## 5. Inspect the results

Keep the full MTEB results and the printed package versions with each experiment.
Use the same task selection and score field when comparing models. The paper's
summary is the unweighted mean across its eight task scores.


In [ ]:
print(results)
